In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [2]:
path_to_dataset = '../../data/processed/online_shoppers_final.csv'

if not os.path.isfile(path_to_dataset):
    print("not good")
else:
    df = pd.read_csv(path_to_dataset)
    

In [3]:
df.shape

(15445, 23)

## Train/Test split + subseturi

In [4]:
y = df['Revenue']
X = df.drop('Revenue', axis=1)
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=7,
    stratify=y
)
 
print(f"\nTrain: {len(X_train):,} înregistrări ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test : {len(X_test):,} înregistrări ({len(X_test)/len(df)*100:.1f}%)")
 
print(f"\nDistribuție Revenue în TRAIN:")
print(f"FALSE (0): {(y_train==0).sum():,} ({(y_train==0).mean()*100:.1f}%)")
print(f"TRUE (1): {(y_train==1).sum():,} ({(y_train==1).mean()*100:.1f}%)")
 
print(f"\nDistribuție Revenue în TEST:")
print(f"FALSE (0): {(y_test==0).sum():,} ({(y_test==0).mean()*100:.1f}%)")
print(f"TRUE (1): {(y_test==1).sum():,} ({(y_test==1).mean()*100:.1f}%)")


Train: 10,811 înregistrări (70.0%)
Test : 4,634 înregistrări (30.0%)

Distribuție Revenue în TRAIN:
FALSE (0): 7,208 (66.7%)
TRUE (1): 3,603 (33.3%)

Distribuție Revenue în TEST:
FALSE (0): 3,089 (66.7%)
TRUE (1): 1,545 (33.3%)


-   Mentinem raportul 2:1 in ambele seturi

## Definirea subseturilor de variabile

In [5]:
df.columns

Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType',
       'Weekend', 'Revenue', 'PageValues_disc', 'BounceRates_disc',
       'ExitRates_disc', 'ProductRelated_Duration_disc',
       'Administrative_Duration_disc'],
      dtype='str')

In [6]:
# Analiza A1 – CART
# Variabile numerice continue -> potrivite pentru CART (criteriu Gini pe valori reale)
# PageValues: cel mai discriminant predictor (corelatie 0.49 cu Revenue)
# BounceRates: comportament de navigare
# VisitorType: nou vs returning visitor
A1_features = ['PageValues', 'BounceRates', 'VisitorType']
print(f"\nA1 (CART) -> {A1_features}")
print(f"Logică: PageValues e cel mai puternic predictor + comportament sesiune")
 
# Analiza A2 – CHAID 
# CHAID necesită variabile CATEGORIALE (test Chi-Square)
# folosim variabilele discretizate + calitative originale
A2_features = ['ExitRates_disc', 'Month', 'SpecialDay', 'Weekend']
print(f"\nA2 (CHAID) -> {A2_features}")
print(f"Logică: variabile temporale + comportament sesiune, toate categoriale")
 
# Analiza A3 – QUEST
# Mix de variabile numerice + categoriale
# TrafficType: sursa de trafic (organic, paid, direct etc.)
# Region: zona geografică
A3_features = ['ProductRelated_Duration_disc', 'Region', 'TrafficType']
print(f"\nA3 (QUEST) -> {A3_features}")
print(f"Logică: comportament de produs + context geografic/trafic")
 
# Analiza A4 – CART complet (bonus)
# Toate variabilele numerice pentru performanță maximă
A4_features = ['PageValues', 'BounceRates', 'ExitRates',
               'ProductRelated_Duration', 'Administrative_Duration',
               'ProductRelated', 'Administrative', 'Informational',
               'Informational_Duration', 'SpecialDay',
               'Month', 'VisitorType', 'Weekend',
               'Region', 'TrafficType', 'OperatingSystems', 'Browser']
print(f"\nA4 (CART complet/bonus) -> toate {len(A4_features)} variabile")
print(f"Logică: performanță maximă + feature importance")


A1 (CART) -> ['PageValues', 'BounceRates', 'VisitorType']
Logică: PageValues e cel mai puternic predictor + comportament sesiune

A2 (CHAID) -> ['ExitRates_disc', 'Month', 'SpecialDay', 'Weekend']
Logică: variabile temporale + comportament sesiune, toate categoriale

A3 (QUEST) -> ['ProductRelated_Duration_disc', 'Region', 'TrafficType']
Logică: comportament de produs + context geografic/trafic

A4 (CART complet/bonus) -> toate 17 variabile
Logică: performanță maximă + feature importance


In [7]:
df.dtypes

Administrative                    int64
Administrative_Duration         float64
Informational                     int64
Informational_Duration          float64
ProductRelated                    int64
ProductRelated_Duration         float64
BounceRates                     float64
ExitRates                       float64
PageValues                      float64
SpecialDay                      float64
Month                             int64
OperatingSystems                  int64
Browser                           int64
Region                            int64
TrafficType                       int64
VisitorType                       int64
Weekend                           int64
Revenue                           int64
PageValues_disc                     str
BounceRates_disc                    str
ExitRates_disc                      str
ProductRelated_Duration_disc        str
Administrative_Duration_disc        str
dtype: object

In [8]:
from sklearn.preprocessing import LabelEncoder

In [9]:
le = LabelEncoder()

subsets = {
    'A1_CART':  A1_features,
    'A2_CHAID': A2_features,
    'A3_QUEST': A3_features,
    'A4_CART_full': A4_features
}
 
train_test_sets = {}
 
for name, features in subsets.items():
    X_tr = X_train[features].copy()
    X_te = X_test[features].copy()
 
    for col in X_tr.columns:
        if X_tr[col].dtype.name == 'category' or X_tr[col].dtype == 'object':
            X_tr[col] = X_tr[col].astype(str)
            X_te[col] = X_te[col].astype(str)
            le = LabelEncoder()
            le.fit(pd.concat([X_tr[col], X_te[col]]))
            X_tr[col] = le.transform(X_tr[col])
            X_te[col] = le.transform(X_te[col])
 
    train_test_sets[name] = (X_tr, X_te, y_train, y_test)
 
    print(f"\n  {name}:")
    print(f"Features : {features}")
    print(f"Train: {X_tr.shape}")
    print(f"Test : {X_te.shape}")
    print(f"Dtypes: {dict(X_tr.dtypes)}")


  A1_CART:
Features : ['PageValues', 'BounceRates', 'VisitorType']
Train: (10811, 3)
Test : (4634, 3)
Dtypes: {'PageValues': dtype('float64'), 'BounceRates': dtype('float64'), 'VisitorType': dtype('int64')}

  A2_CHAID:
Features : ['ExitRates_disc', 'Month', 'SpecialDay', 'Weekend']
Train: (10811, 4)
Test : (4634, 4)
Dtypes: {'ExitRates_disc': <StringDtype(storage='python', na_value=nan)>, 'Month': dtype('int64'), 'SpecialDay': dtype('float64'), 'Weekend': dtype('int64')}

  A3_QUEST:
Features : ['ProductRelated_Duration_disc', 'Region', 'TrafficType']
Train: (10811, 3)
Test : (4634, 3)
Dtypes: {'ProductRelated_Duration_disc': <StringDtype(storage='python', na_value=nan)>, 'Region': dtype('int64'), 'TrafficType': dtype('int64')}

  A4_CART_full:
Features : ['PageValues', 'BounceRates', 'ExitRates', 'ProductRelated_Duration', 'Administrative_Duration', 'ProductRelated', 'Administrative', 'Informational', 'Informational_Duration', 'SpecialDay', 'Month', 'VisitorType', 'Weekend', 'Region

In [10]:
import joblib
joblib.dump(train_test_sets, '../../data/processed/train_test_sets.pkl')

['../../data/processed/train_test_sets.pkl']